# Homework 04: Evaluation

Evaluate keyword, vector, and hybrid search using a ground truth dataset.

## Setup

In [1]:
from gitsource import GithubRepositoryDataReader, chunk_documents
from minsearch import Index, VectorSearch
from embedder import Embedder
from evaluation_utils import llm_structured_retry
from pydantic import BaseModel
from dotenv import load_dotenv
from openai import OpenAI
from tqdm.auto import tqdm
import pandas as pd
import numpy as np
import json

### Load documents

In [2]:
reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
documents = [file.parse() for file in reader.read()]
print(f"Loaded {len(documents)} documents")

Loaded 72 documents


### Chunk documents

In [3]:
chunks = chunk_documents(documents, size=2000, step=1000)
print(f"Created {len(chunks)} chunks")

Created 295 chunks


### Setup LLM client

In [4]:
load_dotenv()
openai_client = OpenAI()

### Build text index

In [5]:
texts = [doc["filename"] + " " + doc["content"] for doc in chunks]

def build_index(documents):
    index = Index(
        text_fields=["content"],
        keyword_fields=["filename"]
    )
    index.fit(documents)
    return index

index = build_index(chunks)

### Build vector index

In [6]:
embed = Embedder()

batch_size = 50
X = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = embed.encode_batch(batch)
    X.extend(batch_vectors)

X = np.array(X)

vindex = VectorSearch(keyword_fields=["filename"])
vindex.fit(X, chunks)

  0%|          | 0/6 [00:00<?, ?it/s]

### Load ground truth data

In [7]:
ground_truth = pd.read_csv('ground-truth.csv')
print(f"Loaded {len(ground_truth)} ground truth questions")
ground_truth.head()

Loaded 360 ground truth questions


,question,filename
0,What exactly is a retrieval-augmented generati...,01-agentic-rag/lessons/01-intro.md
1,Why does this course build the RAG project in ...,01-agentic-rag/lessons/01-intro.md
2,What are the main weaknesses of large language...,01-agentic-rag/lessons/01-intro.md
3,What will the course build in the first part o...,01-agentic-rag/lessons/01-intro.md
4,What kind of example app are you building here...,01-agentic-rag/lessons/01-intro.md


## Q1. Generating questions

Generate questions for the first 3 pages and compute average input tokens.

In [8]:
class Questions(BaseModel):
    questions: list[str]

data_gen_instructions = """
You emulate a student who is taking our LLM course.
You are given one lesson page from the course.
Formulate 5 questions this student might ask that are answered by this page.

Rules:
- The page should contain the answer to each question.
- Make the questions complete and not too short.
- Use as few words as possible from the page; don't copy its phrasing.
- The questions should resemble how people actually ask things online:
  not too formal, not too short, not too long.
- Ask about the content of the lesson, not about its formatting or filename.
""".strip()

def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)
    out, usage = llm_structured_retry(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )
    results = []
    for q in out.questions:
        results.append({
            "question": q,
            "document": doc["filename"]
        })
    return results, usage

In [9]:
gen_ground_truth = []
usages = []

for doc in tqdm(documents[:3]):
    records, usage = generate_ground_truth(doc)
    gen_ground_truth.extend(records)
    usages.append(usage)

  0%|          | 0/3 [00:00<?, ?it/s]

In [10]:
avg_input_tokens = (
    usages[0].input_tokens
    + usages[1].input_tokens
    + usages[2].input_tokens
) / 3
avg_input_tokens

1353.0

## Search functions

Define `text_search`, `vector_search`, `hybrid_search`, and evaluation helpers.

In [11]:
def text_search(query, num_results=5):
    return index.search(query, num_results=num_results)

def vector_search(query, num_results=5):
    query_vector = embed.encode(query)
    return vindex.search(query_vector, num_results=num_results)

In [12]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}
    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc
    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

def hybrid_search(query, k=60):
    text_results = text_search(query, num_results=10)
    vector_results = vector_search(query, num_results=10)
    return rrf([text_results, vector_results], k=k)

### Evaluation metrics

In [13]:
def compute_relevance(q, search_function):
    doc_id = q["filename"]
    results = search_function(query=q["question"])
    relevance = []
    for d in results:
        relevance.append(int(d["filename"] == doc_id))
    return relevance

def compute_relevance_total(ground_truth, search_function):
    relevance_total = []
    for _, q in ground_truth.iterrows():
        relevance = compute_relevance(q, search_function)
        relevance_total.append(relevance)
    return relevance_total

def hit_rate(relevance):
    cnt = 0
    for line in relevance:
        if 1 in line:
            cnt = cnt + 1
    return cnt / len(relevance)

def mrr(relevance):
    total_score = 0.0
    for line in relevance:
        for rank in range(len(line)):
            if line[rank] == 1:
                total_score = total_score + 1 / (rank + 1)
                break
    return total_score / len(relevance)

def evaluate(ground_truth, search_function):
    relevance_total = compute_relevance_total(ground_truth, search_function)
    return {
        "hit_rate": hit_rate(relevance_total),
        "mrr": mrr(relevance_total),
    }

## Q2. First result with text search

In [14]:
q = ground_truth["question"][0]
results = text_search(q, num_results=5)
results[0]["filename"]

'01-agentic-rag/lessons/03-rag.md'

## Q3. First result with vector search

In [15]:
results = vector_search(q, num_results=5)
results[0]["filename"]

'01-agentic-rag/lessons/01-intro.md'

## Q4. Evaluating text search

In [16]:
result = evaluate(ground_truth, text_search)
result

{'hit_rate': 0.7583333333333333, 'mrr': 0.5942592592592594}

## Q5. Evaluating vector search

In [17]:
result = evaluate(ground_truth, vector_search)
result

{'hit_rate': 0.7277777777777777, 'mrr': 0.5469907407407407}

## Q6. Tuning hybrid search

Evaluate `hybrid_search` for k values 1, 50, 100, 200.

In [18]:
for k_param in [1, 50, 100, 200]:
    result = evaluate(
        ground_truth,
        lambda query, k=k_param: hybrid_search(query, k_param)
    )
    print(f"k={k_param}: {result}")

k=1: {'hit_rate': 0.8194444444444444, 'mrr': 0.6486111111111111}
k=50: {'hit_rate': 0.8361111111111111, 'mrr': 0.6418981481481484}
k=100: {'hit_rate': 0.8361111111111111, 'mrr': 0.6418981481481484}
k=200: {'hit_rate': 0.8361111111111111, 'mrr': 0.6418981481481484}
